# Train GBM Models
Trains GBM Global and GBM Segmented (42 submodels) on `gold_model.csv`.
Exports artifacts to `models/gbm/`.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
import warnings; warnings.filterwarnings('ignore')
from config import GOLD_MODEL, GBM_MODELS_DIR

TARGET     = 'default_12m'
GROUP_COLS = ['customer_segment', 'industry_sector']
MODEL_LABEL = 'GBM'
OUT_DIR     = GBM_MODELS_DIR


In [2]:
df = pd.read_csv(GOLD_MODEL)
print(f'Rows: {len(df):,}  Cols: {df.shape[1]}')
print(f'Default rate: {df[TARGET].mean()*100:.1f}%')
df.head(2)


Rows: 24,974  Cols: 35
Default rate: 16.6%


,source_system,ocr_engine,doc_type,document_image_quality,ocr_confidence,ocr_error_count,text_language,normalized_amount,normalization_method,match_score,...,customer_segment,industry_sector,credit_requested_value,income_declared,tenure_months,collateral_type,ltv,pd_model_score,default_12m,env_risk_level
0,OCR_PDF,AZURE_OCR,EXTRATO,0.821,0.562,4,PT,10348.78,RULES_V1,0.720,...,PJ_ME,PF,10898.86,53421.80,210,IMOVEL,0.67,0.297,0,BAIXO
1,EMAIL_ATTACH,GOOGLE_VISION,EXTRATO,0.622,0.830,1,PT,21423.14,RULES_V2,0.659,...,PF,INDUSTRIA,26290.89,28138.01,152,VEICULO,0.67,0.223,1,BAIXO


In [3]:
# Label-encode categoricals except GROUP_COLS (kept as strings for routing)
df_enc = df.copy()
encoders = {}
for col in [c for c in df_enc.select_dtypes(include=['object']).columns
            if c not in GROUP_COLS]:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))
    encoders[col] = le

df_enc['_seg'] = df['customer_segment']
df_enc['_sec'] = df['industry_sector']

FEATURE_COLS = [c for c in df_enc.columns
                if c not in [TARGET] + GROUP_COLS + ['_seg', '_sec']]

# Within-group stratified 80/20 split
train_parts, test_parts = [], []
for (seg, sec), grp in df_enc.groupby(['_seg', '_sec']):
    if grp[TARGET].nunique() < 2:
        train_parts.append(grp)
        continue
    tr, te = train_test_split(grp, test_size=0.2, random_state=42, stratify=grp[TARGET])
    train_parts.append(tr)
    test_parts.append(te)

df_train = pd.concat(train_parts).reset_index(drop=True)
df_test  = pd.concat(test_parts).reset_index(drop=True)
print(f'Train: {len(df_train):,}  Test: {len(df_test):,}')


Train: 19,963  Test: 5,011


In [4]:
# Encode GROUP_COLS for the global model (submodels use FEATURE_COLS only)
group_le = {}
for col in GROUP_COLS:
    le_g = LabelEncoder()
    le_g.fit(df[col].astype(str))
    group_le[col] = le_g

def _with_groups(df_split):
    d = df_split[FEATURE_COLS + GROUP_COLS].copy()
    for col in GROUP_COLS:
        d[col] = group_le[col].transform(df_split[col].astype(str))
    return d

X_tr_g = _with_groups(df_train);  y_tr = df_train[TARGET]
X_te_g = _with_groups(df_test);   y_te = df_test[TARGET]
FEATURE_COLS_GLOBAL = X_tr_g.columns.tolist()


## Train GBM Global

In [5]:
model_global = HistGradientBoostingClassifier(
    max_iter=300, max_depth=4, learning_rate=0.05,
    class_weight='balanced', random_state=42
)
model_global.fit(X_tr_g, y_tr)
proba_global = model_global.predict_proba(X_te_g)[:, 1]
print(f'GBM Global ROC-AUC: {roc_auc_score(y_te, proba_global):.4f}')


GBM Global ROC-AUC: 0.5845


## Train GBM Segmented (42 submodels)

In [6]:
GBM_PARAMS = dict(max_iter=200, max_depth=4, learning_rate=0.05,
                  class_weight='balanced', random_state=42)
models_seg = {}
for (seg, sec), grp in df_train.groupby(['_seg', '_sec']):
    X, y = grp[FEATURE_COLS], grp[TARGET]
    if y.nunique() < 2: continue
    gbm = HistGradientBoostingClassifier(**GBM_PARAMS)
    gbm.fit(X, y)
    models_seg[(seg, sec)] = gbm
    print(f'  [{seg:15s}|{sec:10s}] n={len(grp)}')
print(f'Done. {len(models_seg)} submodels trained.')


  [AGRO_GRANDE    |COMERCIO  ] n=478


  [AGRO_GRANDE    |GOVERNO   ] n=464


  [AGRO_GRANDE    |INDUSTRIA ] n=492


  [AGRO_GRANDE    |PF        ] n=458


  [AGRO_GRANDE    |RURAL     ] n=507


...

  [PJ_GRANDE      |RURAL     ] n=446


  [PJ_GRANDE      |SERVICOS  ] n=487


  [PJ_ME          |COMERCIO  ] n=468
  [PJ_ME          |GOVERNO   ] n=459


  [PJ_ME          |INDUSTRIA ] n=499
  [PJ_ME          |PF        ] n=468


  [PJ_ME          |RURAL     ] n=486
  [PJ_ME          |SERVICOS  ] n=496
Done. 42 submodels trained.


In [7]:
# Evaluate segmented models on df_test (same hold-out as global model)
seg_col = df_test['_seg'].values
sec_col = df_test['_sec'].values
y_te_arr = y_te.values
X_seg_te = df_test[FEATURE_COLS].values
proba_seg = np.zeros(len(df_test))

for (seg, sec), m in models_seg.items():
    mask = (seg_col == seg) & (sec_col == sec)
    if mask.sum():
        proba_seg[mask] = m.predict_proba(X_seg_te[mask])[:, 1]

mask_any = proba_seg > 0
seg_auc = roc_auc_score(y_te_arr[mask_any], proba_seg[mask_any])
print(f'Global   ROC-AUC: {roc_auc_score(y_te, proba_global):.4f}')
print(f'Segmented ROC-AUC: {seg_auc:.4f}')


Global   ROC-AUC: 0.5845
Segmented ROC-AUC: 0.5140


In [8]:
seg_metricas = {}
for seg_val in df['customer_segment'].unique():
    m = seg_col == seg_val
    y_s, p_s = y_te_arr[m], proba_seg[m]
    if len(y_s) > 10 and y_s.sum() > 1 and p_s.sum() > 0:
        seg_metricas[seg_val] = {
            'roc': round(roc_auc_score(y_s, p_s), 4),
            'pr':  round(average_precision_score(y_s, p_s), 4),
            'n':   int(m.sum()),
        }

def _sample_roc(fpr, tpr, n=60):
    idx = np.linspace(0, len(fpr)-1, n, dtype=int)
    return fpr[idx].tolist(), tpr[idx].tolist()

metricas = {
    f'{MODEL_LABEL} Global': {
        'roc': round(roc_auc_score(y_te, proba_global), 4),
        'pr':  round(average_precision_score(y_te, proba_global), 4),
    },
    f'{MODEL_LABEL} Segmentado': {
        'roc': round(roc_auc_score(y_te_arr[mask_any], proba_seg[mask_any]), 4),
        'pr':  round(average_precision_score(y_te_arr[mask_any], proba_seg[mask_any]), 4),
    },
}

roc_curves = {}
for label, proba_e, tgt_e in [
    (f'{MODEL_LABEL} Global',    proba_global,         y_te.values),
    (f'{MODEL_LABEL} Segmentado', proba_seg[mask_any], y_te_arr[mask_any]),
]:
    fpr, tpr, _ = roc_curve(tgt_e, proba_e)
    roc_curves[label] = _sample_roc(fpr, tpr)

print(pd.DataFrame(metricas).T.to_string())


                   roc      pr
GBM Global      0.5845  0.2157
GBM Segmentado  0.5140  0.1721


In [9]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'seg').mkdir(exist_ok=True)

le_dict_global = {**encoders, **group_le}
medians_global = X_tr_g.median(numeric_only=True).to_dict()
medians_seg    = {col: float(df_train[col].median()) for col in FEATURE_COLS
                  if pd.api.types.is_numeric_dtype(df_train[col])}
pd_q75 = float(df['pd_model_score'].quantile(0.75))

to_dump = [
    ({'model': model_global, 'le_dict': le_dict_global,
      'feature_cols': FEATURE_COLS_GLOBAL,
      'medians': medians_global, 'pd_score_q75': pd_q75},
     OUT_DIR / 'global.joblib'),
    ({'metricas': metricas, 'seg_metricas': seg_metricas, 'roc_curves': roc_curves},
     OUT_DIR / 'metrics.joblib'),
    ({'encoders': encoders, 'feature_cols': FEATURE_COLS, 'medians': medians_seg},
     OUT_DIR / 'seg_artifacts.joblib'),
] + [(m, OUT_DIR / 'seg' / f'{seg}_{sec}.joblib') for (seg, sec), m in models_seg.items()]

from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor() as pool:
    list(pool.map(lambda a: joblib.dump(a[0], a[1]), to_dump))

print(f'Exported {len(to_dump)} files → {OUT_DIR}')
for k, v in metricas.items():
    print(f'  {k}: ROC={v["roc"]:.4f}  PR={v["pr"]:.4f}')


Exported 45 files → /home/paolot/Workspace/BB_data_analysis/models/gbm
  GBM Global: ROC=0.5845  PR=0.2157
  GBM Segmentado: ROC=0.5140  PR=0.1721
